# Modelo de fraude

Este notebook contiene la carga de datos, generacion de features, entrenamiento, busqueda de hiperparametros y exportacion de modelos.

In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Sequence
import json
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", 200)

In [2]:
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DATA_RAW = ROOT / "data" / "raw"
DATA_SYNTH = ROOT / "data" / "synthetic"
DATA_PROCESSED = ROOT / "data" / "processed"
ARTIFACT = ROOT / "artifact"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ARTIFACT.mkdir(parents=True, exist_ok=True)

from scripts.features.build_features import build_features
from scripts.rules.fraud_rules import integrate_rules_with_features

In [3]:
def load_tables(data_dir: Path = DATA_RAW) -> dict[str, pd.DataFrame]:
    """Carga tablas desde el notebook para evitar hardcodeo dentro de scripts."""
    files = {
        "siniestros": data_dir / "siniestros.csv",
        "asegurados": data_dir / "asegurados.csv",
        "polizas": data_dir / "polizas.csv",
        "documentos": data_dir / "documentos.csv",
        "beneficiarios_proveedores": data_dir / "beneficiarios_proveedores.csv",
        "vehiculos": data_dir / "vehiculos.csv",
    }
    fallback = {
        "siniestros": DATA_SYNTH / "siniestros_100_registros.csv",
        "asegurados": DATA_SYNTH / "asegurados_sinteticos.csv",
        "polizas": DATA_SYNTH / "polizas.csv",
        "documentos": DATA_SYNTH / "documentos.csv",
        "beneficiarios_proveedores": DATA_SYNTH / "beneficiarios_proveedores.csv",
        "vehiculos": DATA_SYNTH / "vehiculos.csv",
    }
    tables = {}
    for name, path in files.items():
        source = path if path.exists() else fallback[name]
        tables[name] = pd.read_csv(source)
    return tables

tables = load_tables()
{name: df.shape for name, df in tables.items()}

{'siniestros': (100, 23),
 'asegurados': (60, 10),
 'polizas': (100, 13),
 'documentos': (295, 10),
 'beneficiarios_proveedores': (35, 9),
 'vehiculos': (28, 11)}

In [4]:
features = build_features(tables)
features = integrate_rules_with_features(features)

features_path = DATA_PROCESSED / "features_siniestros.csv"
features.to_csv(features_path, index=False)

features.shape, features_path

((100, 95),
 PosixPath('/mnt/data/fraud_work/fraudIA-Novo-2S1R-master/data/processed/features_siniestros.csv'))

In [5]:
TARGET_COL = "etiqueta_fraude_simulada"
ID_AND_TEXT_COLS = {
    "id_siniestro", "id_poliza", "id_asegurado", "id_proveedor", "id_vehiculo",
    "placa", "chasis", "motor", "descripcion", "docs_observaciones",
    "ids_siniestros_similares_top5", "ids_siniestros_similares_top5_json",
    "alertas_score_activadas", "reglas_criticas_activadas", "reglas_criticas_activadas_json",
}


def prepare_training_data(df: pd.DataFrame, target_col: str = TARGET_COL):
    if target_col not in df.columns:
        raise ValueError(f"No existe la columna objetivo: {target_col}")
    y = pd.to_numeric(df[target_col], errors="coerce").fillna(0).astype(int)
    X = df.drop(columns=[target_col])
    drop_cols = set(ID_AND_TEXT_COLS)
    drop_cols.update([c for c in X.columns if c.startswith("created_at") or c.startswith("updated_at")])
    drop_cols.update([c for c in X.columns if np.issubdtype(X[c].dtype, np.datetime64)])
    drop_cols.update([c for c in X.columns if c.startswith("fecha_")])
    X = X.drop(columns=[c for c in drop_cols if c in X.columns], errors="ignore")
    for col in X.columns:
        if X[col].dtype == bool:
            X[col] = X[col].astype(int)
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    numeric_cols = X.columns.difference(categorical_cols).tolist()
    return X, y, categorical_cols, numeric_cols


def build_preprocess(categorical_cols: list[str], numeric_cols: list[str]) -> ColumnTransformer:
    try:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    categorical_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)])
    return ColumnTransformer(
        transformers=[("num", numeric_pipeline, numeric_cols), ("cat", categorical_pipeline, categorical_cols)],
        remainder="drop",
    )


@dataclass
class ModelConfig:
    name: str
    estimator: object
    param_distributions: dict


def default_model_configs(random_state: int = 42) -> list[ModelConfig]:
    return [
        ModelConfig(
            name="logistic_regression",
            estimator=LogisticRegression(max_iter=2000, class_weight="balanced"),
            param_distributions={"model__C": np.logspace(-3, 2, 10), "model__solver": ["lbfgs", "liblinear"]},
        ),
        ModelConfig(
            name="decision_tree",
            estimator=DecisionTreeClassifier(class_weight="balanced", random_state=random_state),
            param_distributions={"model__max_depth": [3, 5, 8, 12, None], "model__min_samples_split": [2, 5, 10], "model__min_samples_leaf": [1, 2, 4]},
        ),
        ModelConfig(
            name="random_forest",
            estimator=RandomForestClassifier(class_weight="balanced", random_state=random_state),
            param_distributions={"model__n_estimators": [50, 100], "model__max_depth": [5, 10, None], "model__min_samples_split": [2, 5, 10], "model__min_samples_leaf": [1, 2, 4]},
        ),
    ]


def train_models(configs: Sequence[ModelConfig], preprocess, X_train, y_train, random_state: int = 42, n_iter: int = 10, cv: int = 3):
    trained_models = {}
    rows = []
    for config in configs:
        pipe = Pipeline(steps=[("preprocess", preprocess), ("model", config.estimator)])
        search = RandomizedSearchCV(
            pipe,
            param_distributions=config.param_distributions,
            n_iter=n_iter,
            scoring="f1",
            cv=cv,
            random_state=random_state,
            n_jobs=1,
            verbose=0,
        )
        search.fit(X_train, y_train)
        trained_models[config.name] = search.best_estimator_
        rows.append({"model": config.name, "best_f1_cv": search.best_score_, "best_params": search.best_params_})
    return trained_models, pd.DataFrame(rows).sort_values("best_f1_cv", ascending=False)


def export_models(models: dict[str, Pipeline], artifact_dir: Path) -> dict[str, Path]:
    saved = {}
    for name, model in models.items():
        path = artifact_dir / f"{name}.pkl"
        joblib.dump(model, path)
        saved[name] = path
    return saved


def build_predictions(models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series, ids: pd.Series | None = None) -> pd.DataFrame:
    rows = []
    for name, model in models.items():
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        for idx, true_value, pred, prob in zip(X_test.index, y_test, y_pred, y_prob):
            rows.append({
                "model": name,
                "row_index": idx,
                "id_siniestro": ids.loc[idx] if ids is not None and idx in ids.index else None,
                "y_true": int(true_value),
                "y_pred": int(pred),
                "y_prob": float(prob),
            })
    return pd.DataFrame(rows)

In [6]:
X, y, cat_cols, num_cols = prepare_training_data(features)

metadata = {
    "target_col": TARGET_COL,
    "model_input_columns": X.columns.tolist(),
    "categorical_cols": cat_cols,
    "numeric_cols": num_cols,
}
with open(ARTIFACT / "model_input_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

X.shape, len(cat_cols), len(num_cols)

((100, 73), 17, 56)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
preprocess = build_preprocess(cat_cols, num_cols)
configs = default_model_configs(random_state=42)
trained_models, results_df = train_models(configs, preprocess, X_train, y_train, random_state=42, n_iter=4, cv=3)
results_df

,model,best_f1_cv,best_params
1,decision_tree,1.000000,"{'model__min_samples_split': 2, 'model__min_sa..."
0,logistic_regression,0.869048,"{'model__solver': 'liblinear', 'model__C': 27...."
2,random_forest,0.555556,"{'model__n_estimators': 100, 'model__min_sampl..."


In [8]:
saved_paths = export_models(trained_models, ARTIFACT)
saved_paths

{'logistic_regression': PosixPath('/mnt/data/fraud_work/fraudIA-Novo-2S1R-master/artifact/logistic_regression.pkl'),
 'decision_tree': PosixPath('/mnt/data/fraud_work/fraudIA-Novo-2S1R-master/artifact/decision_tree.pkl'),
 'random_forest': PosixPath('/mnt/data/fraud_work/fraudIA-Novo-2S1R-master/artifact/random_forest.pkl')}

In [9]:
preds = build_predictions(trained_models, X_test, y_test, ids=features["id_siniestro"] if "id_siniestro" in features.columns else None)
preds_path = DATA_PROCESSED / "predictions.csv"
preds.to_csv(preds_path, index=False)
preds.head()

,model,row_index,id_siniestro,y_true,y_pred,y_prob
0,logistic_regression,25,26,0,0,0.002301
1,logistic_regression,17,18,0,0,0.001625
2,logistic_regression,93,94,0,0,0.000103
3,logistic_regression,76,77,1,1,0.999993
4,logistic_regression,1,2,0,0,0.045240


In [10]:
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, digits=3, zero_division=0))

=== logistic_regression ===
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        17
           1      1.000     1.000     1.000         3

    accuracy                          1.000        20
   macro avg      1.000     1.000     1.000        20
weighted avg      1.000     1.000     1.000        20

=== decision_tree ===
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        17
           1      1.000     1.000     1.000         3

    accuracy                          1.000        20
   macro avg      1.000     1.000     1.000        20
weighted avg      1.000     1.000     1.000        20

=== random_forest ===
              precision    recall  f1-score   support

           0      1.000     1.000     1.000        17
           1      1.000     1.000     1.000         3

    accuracy                          1.000        20
   macro avg      1.000     1.000     1.000        20
we